In [0]:
from pyspark.sql.functions import col, to_date, expr, regexp_replace

bronze_df = spark.read.table("workspace.stock_data.stock_data_bronze")

silver_df = bronze_df.select('symbol', 'open', 'close', 'adj_open', 'adj_close', 'volume', 'adj_volume', 'date')

silver_df = (
    silver_df
    .withColumn("open", expr("try_cast(open as double)"))
    .withColumn("close", expr("try_cast(close as double)"))
    .withColumn("adj_open", expr("try_cast(adj_open as double)"))
    .withColumn("adj_close", expr("try_cast(adj_close as double)"))
    .withColumn("volume", expr("try_cast(regexp_replace(volume, '.0$', '') as long)"))
    .withColumn("adj_volume", expr("try_cast(regexp_replace(adj_volume, '.0$', '') as long)"))
    .withColumn("date", to_date(regexp_replace(col("date"), "T.*", ""), "yyyy-MM-dd"))
)

silver_df = silver_df.dropDuplicates(["symbol", "date"])

silver_df.write.format("delta").mode("overwrite").saveAsTable('workspace.stock_data.stock_data_silver')

In [0]:
%sql
SELECT *
FROM workspace.stock_data.stock_data_silver
ORDER BY symbol, date